# Part 2: Creating Vector Embeddings

In this notebook, we'll learn:
1. What embeddings are
2. How to generate embeddings from text
3. How to measure similarity between embeddings
4. How to compare local vs OpenAI embeddings

## Import Libraries

In [ ]:
import sys
sys.path.append('..')

from src.embeddings import EmbeddingGenerator
from src.pdf_loader import load_sample_documents
import numpy as np

print("✓ Libraries imported")

## What are Embeddings?

Embeddings convert text into vectors (lists of numbers) that capture semantic meaning.

In [ ]:
# Initialize embedding generator (local model, free)
generator = EmbeddingGenerator(embedding_type="local")

# Get embedding dimension
dim = generator.get_embedding_dimension()
print(f"Embedding dimension: {dim}")
print(f"This means each piece of text becomes a vector of {dim} numbers")

## Generate Single Embeddings

In [ ]:
# Generate embedding for a single text
text = "Machine learning is a subset of artificial intelligence"
embedding = generator.embed_text(text)

print(f"Text: {text}")
print(f"\nEmbedding vector (first 10 values):")
print(embedding[:10])
print(f"\nFull embedding length: {len(embedding)}")
print(f"Vector type: {type(embedding)}")

## Generate Multiple Embeddings

Let's generate embeddings for multiple texts and see how they compare.

In [ ]:
# Sample texts for embedding
texts = [
    "Machine learning is artificial intelligence",
    "Deep learning uses neural networks",
    "Natural language processing handles text",
    "Computer vision analyzes images",
    "Artificial intelligence and machine learning are related"
]

# Generate embeddings
embeddings = generator.embed_texts(texts)

print(f"Generated {len(embeddings)} embeddings")
for i, (text, emb) in enumerate(zip(texts, embeddings)):
    print(f"  {i+1}. {text} → {len(emb)} dimensions")

## Similarity Between Embeddings

Similar texts should have similar embeddings. Let's measure similarity using cosine similarity.

In [ ]:
# Calculate similarity matrix
print("Similarity Matrix:")
print("(Values closer to 1.0 = more similar)\n")

print("\t", end="")
for i in range(1, 6):
    print(f"T{i:>6}", end="")
print()

for i, emb1 in enumerate(embeddings):
    print(f"T{i+1}", end="\t")
    for j, emb2 in enumerate(embeddings):
        sim = generator.similarity(emb1, emb2)
        print(f"{sim:.3f}", end="  ")
    print()

## Analyze Similarity Patterns

In [ ]:
# Find most similar pairs
print("Most Similar Text Pairs:")
print()

similarities = []
for i in range(len(embeddings)):
    for j in range(i+1, len(embeddings)):
        sim = generator.similarity(embeddings[i], embeddings[j])
        similarities.append((i, j, sim))

# Sort by similarity
similarities.sort(key=lambda x: x[2], reverse=True)

for i, j, sim in similarities[:5]:
    print(f"Similarity: {sim:.4f}")
    print(f"  Text {i+1}: {texts[i]}")
    print(f"  Text {j+1}: {texts[j]}")
    print()

## Embedding with Sample Documents

In [ ]:
# Load sample documents
documents = load_sample_documents()

# Extract text from documents
doc_texts = [doc['text'][:500] for doc in documents]  # Use first 500 chars

# Generate embeddings
doc_embeddings = generator.embed_texts(doc_texts)

print(f"Generated embeddings for {len(doc_texts)} documents")

# Calculate similarity between documents
sim = generator.similarity(doc_embeddings[0], doc_embeddings[1])
print(f"\nSimilarity between documents: {sim:.4f}")

## Embedding Quality Metrics

Let's analyze the properties of our embeddings.

In [ ]:
# Analyze embedding statistics
embeddings_array = np.array(embeddings)

print("Embedding Statistics:")
print(f"  Shape: {embeddings_array.shape}")
print(f"  Mean value: {np.mean(embeddings_array):.6f}")
print(f"  Std deviation: {np.std(embeddings_array):.6f}")
print(f"  Min value: {np.min(embeddings_array):.6f}")
print(f"  Max value: {np.max(embeddings_array):.6f}")

# Calculate norms
norms = np.linalg.norm(embeddings_array, axis=1)
print(f"\nEmbedding Norms (magnitude):")
for i, norm in enumerate(norms):
    print(f"  Text {i+1}: {norm:.4f}")

## Performance Considerations

Let's test the speed of embedding generation.

In [ ]:
import time

# Test batch embedding speed
num_texts = 100
test_texts = ["Sample text number " + str(i) for i in range(num_texts)]

start_time = time.time()
batch_embeddings = generator.embed_texts(test_texts, batch_size=32)
elapsed = time.time() - start_time

print(f"Embedded {num_texts} texts in {elapsed:.2f} seconds")
print(f"Speed: {num_texts/elapsed:.1f} texts/second")
print(f"Per-text time: {elapsed/num_texts*1000:.2f} ms")

## Summary

Key learnings from this notebook:
- ✓ Embeddings convert text into numerical vectors
- ✓ Similar texts produce similar embeddings
- ✓ Cosine similarity measures the relationship between embeddings
- ✓ Local embeddings are fast and free
- ✓ Embeddings are normalized vectors with specific properties

**Next steps:**
- Continue to notebook 3 to learn about vector databases
- Or experiment with different embedding models